# Transformer 超参数搜索

在 ETTh1 h96 上做 128 组合网格搜索，选出 val_loss 最优配置。

搜索空间：d_model × nhead × num_layers × dim_feedforward × dropout × lr × weight_decay = 2^7 = 128 组合

固定参数：epochs=25, patience=5, batch_size=32, seed=216

In [1]:
import sys
from pathlib import Path
import json

# 定位项目根目录：向上查找包含 scripts/ 和 models/ 的目录
_here = Path().resolve()
PROJECT_ROOT = _here
for _ in range(5):
    if (PROJECT_ROOT / 'scripts').is_dir() and (PROJECT_ROOT / 'models').is_dir():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')

from scripts.tune_transformer import (
    build_search_grid, run_trial, write_summary, set_seed, count_params
)
from models import TransformerModel, TimeSeriesDataset
from models.trainer import resolve_device
import torch

ROOT = PROJECT_ROOT

# ========== 搜索配置（内嵌，与 configs/transformer_search.json 同步） ==========
SEARCH_SPACE = {
    "d_model":         [64, 128],
    "nhead":           [4, 8],
    "num_layers":      [2, 3],
    "dim_feedforward": [128, 256],
    "dropout":         [0.05, 0.1],
    "lr":              [1e-4, 5e-5],
    "weight_decay":    [0.0, 1e-5],
}

FIXED = {
    "epochs":       25,
    "patience":     5,
    "batch_size":   32,
    "seed":         216,
    "sample_limit": 0,
    "device":       "auto",
    "num_workers":  0,
}

DATASETS = ["ETTh1"]
HORIZONS = [96]
print(f'Search space: {len(SEARCH_SPACE)} params, grid size: {len(build_search_grid(SEARCH_SPACE))}')

Project root: /root/demo



libgomp: Invalid value for environment variable OMP_NUM_THREADS


Search space: 7 params, grid size: 128


## 1. 查看搜索空间

In [2]:
grid = build_search_grid(SEARCH_SPACE)
print(f'搜索空间大小: {len(grid)} 组合')
print(f'前3组:')
for i, combo in enumerate(grid[:3]):
    print(f'  {i+1}. {combo}')

搜索空间大小: 128 组合
前3组:
  1. {'d_model': 64, 'nhead': 4, 'num_layers': 2, 'dim_feedforward': 128, 'dropout': 0.05, 'lr': 0.0001, 'weight_decay': 0.0}
  2. {'d_model': 64, 'nhead': 4, 'num_layers': 2, 'dim_feedforward': 128, 'dropout': 0.05, 'lr': 0.0001, 'weight_decay': 1e-05}
  3. {'d_model': 64, 'nhead': 4, 'num_layers': 2, 'dim_feedforward': 128, 'dropout': 0.05, 'lr': 5e-05, 'weight_decay': 0.0}


## 2. Dry-run：验证参数量范围

In [3]:
data_dir = ROOT / 'data' / 'processed'
ds = TimeSeriesDataset(data_dir, 'ETTh1', 96, 'train')
input_size = ds.input_size
print(f'ETTh1 h96: input_size={input_size}, target_idx={ds.target_idx}')

param_counts = []
for combo in grid:
    model = TransformerModel(
        input_size=input_size, horizon=96,
        d_model=combo['d_model'], nhead=combo['nhead'],
        num_layers=combo['num_layers'],
        dim_feedforward=combo['dim_feedforward'],
        dropout=combo['dropout']
    )
    params = count_params(model)
    param_counts.append((combo, params))

params_only = [p for _, p in param_counts]
print(f'参数量范围: {min(params_only):,} ~ {max(params_only):,}')
print(f'中位数: {sorted(params_only)[len(params_only)//2]:,}')

# 显示参数量最大和最小的配置
min_combo = min(param_counts, key=lambda x: x[1])
max_combo = max(param_counts, key=lambda x: x[1])
print(f'\n最小: {min_combo[1]:,} params  {min_combo[0]}')
print(f'最大: {max_combo[1]:,} params  {max_combo[0]}')

加载 ETTh1 train 数据: X=torch.Size([8449, 96, 7]), Y=torch.Size([8449, 96, 7])
ETTh1 h96: input_size=7, target_idx=6
参数量范围: 115,296 ~ 501,664
中位数: 303,392

最小: 115,296 params  {'d_model': 64, 'nhead': 4, 'num_layers': 2, 'dim_feedforward': 128, 'dropout': 0.05, 'lr': 0.0001, 'weight_decay': 0.0}
最大: 501,664 params  {'d_model': 128, 'nhead': 4, 'num_layers': 3, 'dim_feedforward': 256, 'dropout': 0.05, 'lr': 0.0001, 'weight_decay': 0.0}


## 3. 运行搜索

可选方式：
- 运行全部 128 组合（约 1h on CUDA）
- 先用 `--max-trials 10` 快速验证
- 或使用下方单元格逐批运行

In [ ]:
# 方式1：直接运行脚本（推荐在终端执行）
# !python ../scripts/tune_transformer.py --config ../configs/transformer_search.json

# 方式2：在此 notebook 中直接运行训练
from scripts.tune_transformer import run_trial, write_summary
import time

output_dir = ROOT / 'test_results' / 'h96' / 'ETTh1' / 'transformer'
output_dir.mkdir(parents=True, exist_ok=True)

fixed = {**FIXED}
data_dir_path = ROOT / 'data' / 'processed'

print(f'Device: {resolve_device(fixed["device"])}')
print(f'Output: {output_dir}')
print(f'Total grid: {len(grid)} configs')
print('=' * 60)

# ========== 开始训练 ==========
results = []
start_time = time.time()

for i, combo in enumerate(grid):
    print(f'\n[{i+1}/{len(grid)}] {combo}')
    result = run_trial(
        dataset_name='ETTh1',
        horizon=96,
        trial_params=combo,
        fixed=fixed,
        data_dir=data_dir_path,
        output_dir=output_dir,
        skip_existing=fixed.get('skip_existing', True),
    )
    if result:
        results.append(result)
        print(f"  -> val_loss={result['best_val_loss']:.4f}, MSE={result['metrics']['MSE']:.4f}, R2={result['metrics']['R2']:.4f}")

# 保存汇总
if results:
    write_summary(results, output_dir)
    elapsed = time.time() - start_time
    print(f'\n{"=" * 60}')
    print(f'完成! 共 {len(results)}/{len(grid)} 组实验, 耗时 {elapsed:.1f}s')
    print(f'结果保存到: {output_dir}')
    
    # 显示最优配置
    best = min(results, key=lambda x: x['best_val_loss'])
    print(f'\n最优配置: val_loss={best["best_val_loss"]:.4f}')
    print(f'  d_model={best["d_model"]}, nhead={best["nhead"]}, num_layers={best["num_layers"]}')
    print(f'  dim_feedforward={best["dim_feedforward"]}, dropout={best["dropout"]}')
    print(f'  lr={best["lr"]}, weight_decay={best["weight_decay"]}')
else:
    print('没有成功完成的实验')

Device: cuda
Output: /root/demo/test_results/h96/ETTh1/transformer
Total grid: 128 configs

[1/128] {'d_model': 64, 'nhead': 4, 'num_layers': 2, 'dim_feedforward': 128, 'dropout': 0.05, 'lr': 0.0001, 'weight_decay': 0.0}
加载 ETTh1 train 数据: X=torch.Size([8449, 96, 7]), Y=torch.Size([8449, 96, 7])
加载 ETTh1 val 数据: X=torch.Size([2689, 96, 7]), Y=torch.Size([2689, 96, 7])
加载 ETTh1 test 数据: X=torch.Size([5709, 96, 7]), Y=torch.Size([5709, 96, 7])
开始训练: ETTh1_h96_transformer_d64_h4_l2_ff128_dp005_lr0.0001_wd0.0
设备: cuda
训练样本: 8449
验证样本: 2689
批次大小: 32
Epoch   1/25 | Train Loss: 0.821263 R²: 0.1663 | Val Loss: 1.350725 R²: -0.0329 ✓ | Time: 2.2s
Epoch   2/25 | Train Loss: 0.580111 R²: 0.4106 | Val Loss: 1.257502 R²: 0.0381 ✓ | Time: 1.3s
Epoch   3/25 | Train Loss: 0.516153 R²: 0.4755 | Val Loss: 1.196071 R²: 0.0858 ✓ | Time: 1.3s
Epoch   4/25 | Train Loss: 0.474221 R²: 0.5171 | Val Loss: 1.144711 R²: 0.1312 ✓ | Time: 1.5s
Epoch   5/25 | Train Loss: 0.438076 R²: 0.5556 | Val Loss: 1.074962 R²: 

## 4. 分析结果

搜索完成后，读取 summary 并分析。

In [ ]:
import pandas as pd

csv_path = ROOT / 'test_results' / 'h96' / 'ETTh1' / 'transformer' / 'transformer_search_summary.csv'
if csv_path.exists():
    df = pd.read_csv(csv_path)
    df = df.sort_values('best_val_loss')
    print(f'共 {len(df)} 组实验')
    print(f'\nTop-5 by val_loss:')
    cols = ['run_name', 'd_model', 'nhead', 'num_layers', 'dim_feedforward',
            'dropout', 'lr', 'weight_decay', 'model_params', 'best_val_loss', 'MSE', 'R2']
    display(df[cols].head())
else:
    print('搜索结果尚未生成，请先运行搜索。')